# BESTEST LBNL-style report — Case 600 pilot

Native Python ISO results may be used for formal ASHRAE annual-energy assessment. `modelica_solar` is a controlled-forcing comparison and is not formal pass/fail.

In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/mpl-bestest-notebook')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

ISO_MODE = 'modelica_solar'  # accepted: 'modelica_solar', 'native'
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / 'scripts/run_case600_pilot.py').is_file()), None)
if root is None:
    candidate = cwd / '2__validation/_BESTEST'
    if not (candidate / 'scripts/run_case600_pilot.py').is_file():
        raise FileNotFoundError('Open from _BESTEST or _development.')
    root = candidate
sys.path.insert(0, str(root / 'scripts'))
from bestest_reporting import (MODELICA_LABEL, annual_energy_table, daily_slice, peak_table, selected_hourly, selected_iso_mode)
subprocess.run([sys.executable, 'scripts/run_case600_pilot.py'], cwd=root, check=True)
reference = root / '_ref/BESTEST_LBNL_all_cases_reference.md'
case = '600'
result_dir = root / 'results' / f'case{case}'
metrics = pd.read_csv(result_dir / f'case{case}_metrics.csv')
hourly = pd.read_csv(result_dir / f'case{case}_feb1_load_profile.csv')
iso_run_mode, ISO_LABEL = selected_iso_mode(ISO_MODE)

## 1. Annual heating energy

Annual heating energy for the implemented conditioned case, in MWh.

In [ ]:
heating = annual_energy_table(metrics, reference, case, 'annual_heating_energy', ISO_MODE)
display(heating)
plot = heating.drop(columns=['Case', 'ASHRAE lower', 'ASHRAE upper', 'Python formal status']).T.rename(columns={0: case})
ax = plot.plot.bar(figsize=(10, 3.2), legend=False, color='#4C78A8')
ax.axhspan(float(heating['ASHRAE lower'].iloc[0]), float(heating['ASHRAE upper'].iloc[0]), color='0.85', zorder=0)
ax.set(ylabel='MWh', title=f'Case {case}: annual heating energy')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## 2. Annual cooling energy

Annual cooling energy for the implemented conditioned case, in MWh.

In [ ]:
cooling = annual_energy_table(metrics, reference, case, 'annual_cooling_energy', ISO_MODE)
display(cooling)
plot = cooling.drop(columns=['Case', 'ASHRAE lower', 'ASHRAE upper', 'Python formal status']).T.rename(columns={0: case})
ax = plot.plot.bar(figsize=(10, 3.2), legend=False, color='#F58518')
ax.axhspan(float(cooling['ASHRAE lower'].iloc[0]), float(cooling['ASHRAE upper'].iloc[0]), color='0.85', zorder=0)
ax.set(ylabel='MWh', title=f'Case {case}: annual cooling energy')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()

## 3. Peak heating load

Peak heating load and completed-hour occurrence time, in kW.

In [ ]:
display(peak_table(metrics, reference, case, 'peak_heating_load', ISO_MODE))
selected = metrics[((metrics.implementation == 'modelica') & (metrics.run_mode == 'native')) | ((metrics.implementation == 'iso13790') & (metrics.run_mode == iso_run_mode))]
peak = selected[selected.metric == 'peak_heating_load'].copy()
peak['run'] = peak.implementation.map({'modelica': MODELICA_LABEL, 'iso13790': ISO_LABEL})
ax = peak.set_index('run').value.plot.bar(figsize=(5.5, 3.2), color=['#4C78A8', '#54A24B'])
ax.set(ylabel='kW', title=f'Case {case}: peak heating load'); plt.xticks(rotation=20, ha='right'); plt.tight_layout()

## 4. Peak cooling load

Peak cooling load and completed-hour occurrence time, in kW.

In [ ]:
display(peak_table(metrics, reference, case, 'peak_cooling_load', ISO_MODE))
peak = selected[selected.metric == 'peak_cooling_load'].copy()
peak['run'] = peak.implementation.map({'modelica': MODELICA_LABEL, 'iso13790': ISO_LABEL})
ax = peak.set_index('run').value.plot.bar(figsize=(5.5, 3.2), color=['#4C78A8', '#54A24B'])
ax.set(ylabel='kW', title=f'Case {case}: peak cooling load'); plt.xticks(rotation=20, ha='right'); plt.tight_layout()

## 5. Daily load profile

Case 600 heating and cooling loads on 1 February.

In [ ]:
profile = daily_slice(selected_hourly(hourly, ISO_MODE), 2, 1)
profile['run'] = profile.implementation.map({'modelica': MODELICA_LABEL, 'iso13790': ISO_LABEL})
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for run, frame in profile.groupby('run'):
    axes[0].plot(frame.hour, frame.heating_load_W / 1000, label=run)
    axes[1].plot(frame.hour, frame.cooling_load_W / 1000, label=run)
axes[0].set(ylabel='Heating (kW)', title='Case 600: 1 February load profile')
axes[1].set(xlabel='Completed hour', ylabel='Cooling (kW)')
for axis in axes: axis.grid(alpha=.25); axis.legend(fontsize=8)
fig.tight_layout()

## Summary

Current Case 600 annual-energy record; the main report uses the selected `ISO_MODE`.

In [ ]:
summary = metrics[metrics.metric.isin(['annual_heating_energy', 'annual_cooling_energy'])].copy()
summary['Run'] = summary.apply(lambda x: MODELICA_LABEL if x.implementation == 'modelica' else ('Native ISO' if x.run_mode == 'native' else 'ISO + Modelica solar'), axis=1)
display(summary.pivot(index='Run', columns='metric', values='value').rename(columns={'annual_heating_energy': 'Heating (MWh)', 'annual_cooling_energy': 'Cooling (MWh)'}).reindex(['Native ISO', 'ISO + Modelica solar', MODELICA_LABEL]))
print('Controlled-forcing ISO is close to Modelica; native ISO remains affected by the unresolved solar-preprocessing discrepancy. The main report uses ISO_MODE =', ISO_MODE)